In [1]:
import duckdb
import pandas as pd

In [3]:
con = duckdb.connect('games.duckdb')

In [4]:
games = con.execute('SELECT * FROM games').df()

In [6]:
external_games = pd.read_csv('external_games.csv', low_memory=False)

In [7]:
external_games

,id,name,created_at,updated_at,uid,category,year,url,game,checksum,countries,platform,media,external_game_source,game_release_format
0,2939936,Post Apocalypse Courier Service - Delivery Dri...,2024-08-16 15:12:42,2025-09-05 01:35:43,3079190,NaN,2025.0,https://store.steampowered.com/app/3079190,323035,0ac117aa-5598-33b2-9acb-d862e5151c9a,NaN,NaN,NaN,1,NaN
1,2980559,Metaneurosis,2025-02-01 21:07:06,2025-03-31 01:43:17,3483990,1.0,2025.0,https://store.steampowered.com/app/3483990,337286,f3ae2a32-cd1e-9bcd-2bce-2e5d1df4fdf3,NaN,NaN,NaN,1,NaN
2,1838331,The Christmas Spirit: Grimm Tales Collector's ...,2019-12-14 11:31:13,2025-09-04 14:30:24,1198070,NaN,2019.0,https://store.steampowered.com/app/1198070,127501,84e63be8-ba4d-b028-11cf-c30b8ac37599,NaN,NaN,NaN,1,NaN
3,1309824,Strong Bad Episode 2: Strong Badia the Free,2018-07-24 21:37:10,2025-09-04 15:05:21,8350,NaN,0.0,https://store.steampowered.com/app/8350,50176,0eb52c80-55c7-6a4f-6581-640470c6729c,NaN,NaN,NaN,1,NaN
4,2974146,Clawed,2025-01-11 04:06:35,2025-09-05 01:37:01,3394840,NaN,2025.0,https://store.steampowered.com/app/3394840,342028,bdd9cae4-d7f8-46f0-a2f0-daa19d4933e7,NaN,NaN,NaN,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
633523,2867778,FNaF 2 Night of Misfits,2023-12-02 22:08:32,2023-12-12 09:56:29,819306,55.0,NaN,http://gamejolt.com/games/FNaF2-NoM/819306,280135,891e80d0-c114-af3b-3651-663e8fe15aad,NaN,NaN,NaN,55,NaN
633524,2612330,Cuckoo Castle,2022-10-14 20:58:12,2025-08-29 02:55:07,87063,55.0,NaN,http://gamejolt.com/games/cuckoo-castle/87063,135241,aaa34e5e-ab47-e28a-9a3d-910c594cbb16,NaN,NaN,NaN,55,NaN
633525,2888694,PolarDread (VR ONLY),2024-02-17 22:04:59,2025-09-09 00:17:12,618442,NaN,NaN,http://gamejolt.com/games/POLARDREAD/618442,298276,999cca4d-20af-b1a5-0446-61f521bec0ac,NaN,NaN,NaN,55,NaN
633526,2950841,Killer Frequency,2024-09-29 22:04:20,2025-10-22 22:22:38,419748,NaN,NaN,http://gamejolt.com/games/KillerFrequency/419748,124257,89157ccb-3f85-600c-53d9-7484af1f386a,NaN,NaN,NaN,55,NaN


In [15]:
filter_external_games = external_games[external_games['external_game_source'] == 1]

In [18]:
merged_games = games.merge(
    filter_external_games[['uid', 'external_game_source', 'game']],
    left_on='id',
    right_on='game')

In [19]:
merged_games

,id,name,genres,franchise,releaseDate,uid,external_game_source,game
0,337507,Adventurous Slime,"[12, 31, 32]",<NA>,2025-04-22,3379150,1,337507
1,157761,Microsoft Flight Simulator X: Steam Edition - ...,[13],<NA>,2018-05-01,643689,1,157761
2,244436,Lurch,[32],<NA>,2024-08-19,2095540,1,244436
3,128168,Sanguine Soul,"[5, 31, 32]",<NA>,2018-12-06,1113850,1,128168
4,348397,Baccarat,"[14, 32]",<NA>,NaT,3590620,1,348397
...,...,...,...,...,...,...,...,...
148575,34207,Talisman: Digital Edition - The Highland,"[12, 15, 32, 35]",<NA>,2015-03-06,267780,1,34207
148576,89219,Stronghold Crusader II: The Templar &The Duke,"[13, 15]",<NA>,2015-07-01,349230,1,89219
148577,8993,Anomaly Defenders,"[15, 32]",<NA>,2014-05-29,294750,1,8993
148578,11261,The Black Mirror,"[2, 31]",<NA>,2003-10-17,292930,1,11261


In [26]:
merged_games['uid'] = merged_games['uid'].astype(str).str.split(',').str[0].astype(int)

In [22]:
con.execute('ALTER TABLE games ADD COLUMN steamId INTEGER');

In [27]:
con.execute('UPDATE games SET steamId = merged_games.uid FROM merged_games WHERE games.id = merged_games.id')

In [28]:
games_new = con.execute('SELECT * FROM games').df()

In [29]:
games_new

,id,name,genres,franchise,releaseDate,steamId
0,330684,Nightmare Kart: The Old Karts,"[10, 33]",<NA>,NaT,<NA>
1,177310,The Undying Beast,"[13, 32]",<NA>,2020-04-13,<NA>
2,337507,Adventurous Slime,"[12, 31, 32]",<NA>,2025-04-22,3379150
3,350392,Rival Species,[5],<NA>,NaT,<NA>
4,63844,Ace wo Nerae!,[14],2330,1993-12-22,<NA>
...,...,...,...,...,...,...
332595,89219,Stronghold Crusader II: The Templar &The Duke,"[13, 15]",<NA>,2015-07-01,349230
332596,154420,Catacombs Pack,[5],<NA>,2013-03-14,<NA>
332597,8993,Anomaly Defenders,"[15, 32]",<NA>,2014-05-29,294750
332598,11261,The Black Mirror,"[2, 31]",<NA>,2003-10-17,292930


In [30]:
con.close()